# Clean IVV Vineyard Area Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/pml_final_project/blob/main/notebooks/ivv_vineyard_area.ipynb)

This notebook cleans the IVV vineyard area file and saves a processed CSV that can be joined to the wine production dataset as an explanatory variable.

## 1. Install Packages

In [ ]:
!pip install -q pandas numpy openpyxl xlrd lxml scikit-learn

## 2. Clone or Update Repository

In [ ]:
from pathlib import Path
from datetime import datetime
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/FernandaChacara/pml_final_project.git"
ROOT_DIR = Path("/content/pml_final_project")

if ROOT_DIR.exists():
    print("Repository already exists. Pulling latest changes...")
    %cd /content/pml_final_project
    !git pull
else:
    !git clone {REPO_URL} /content/pml_final_project
    %cd /content/pml_final_project

RAW_XLS = ROOT_DIR / "data" / "raw" / "Area_total_de_vinha_PT_2025.xls"
RAW_XLSX = ROOT_DIR / "data" / "raw" / "Area_total_de_vinha_PT_2025.xlsx"
OUTPUT_FILE = ROOT_DIR / "data" / "processed" / "vineyard_area_by_region_clean.csv"

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Root directory:", ROOT_DIR)
print("Raw XLS file:", RAW_XLS)
print("Raw XLSX file:", RAW_XLSX)
print("Output file:", OUTPUT_FILE)

## 3. Prepare and Read the Raw File

The workbook is an `.xls` file. This notebook prefers a converted `.xlsx` file if one exists, but can also read the `.xls` directly with `xlrd`.

In [ ]:
def try_convert_xls_to_xlsx(raw_xls: Path, raw_xlsx: Path) -> Path | None:
    if raw_xlsx.exists():
        print(f"Using existing XLSX file: {raw_xlsx}")
        return raw_xlsx

    if not raw_xls.exists():
        raise FileNotFoundError(
            f"Neither {raw_xlsx.name} nor {raw_xls.name} was found in data/raw."
        )

    print("Converted XLSX file was not found. Trying LibreOffice conversion...")

    try:
        if shutil.which("libreoffice") is None:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "libreoffice"], check=True)

        result = subprocess.run(
            [
                "libreoffice",
                "--headless",
                "--convert-to",
                "xlsx",
                "--outdir",
                str(raw_xls.parent),
                str(raw_xls),
            ],
            capture_output=True,
            text=True,
        )

        if result.returncode == 0 and raw_xlsx.exists():
            print(f"Created XLSX file: {raw_xlsx}")
            return raw_xlsx

        print("LibreOffice conversion did not create an XLSX file. Falling back to direct XLS read.")
        print(result.stderr)
        return None
    except Exception as error:
        print("LibreOffice conversion failed. Falling back to direct XLS read.")
        print(error)
        return None


EXCEL_FILE = try_convert_xls_to_xlsx(RAW_XLS, RAW_XLSX)

if EXCEL_FILE is not None:
    raw_df = pd.read_excel(EXCEL_FILE, sheet_name=0, header=None, dtype=object, engine="openpyxl")
else:
    raw_df = pd.read_excel(RAW_XLS, sheet_name=0, header=None, dtype=object, engine="xlrd")

print("Raw shape:", raw_df.shape)
display(raw_df.head(12))

## 4. Define Cleaning Helpers

In [ ]:
REGION_MAP = {
    "Verdes": "Verdes",
    "Trás-os Montes": "T. Montes",
    "Tras-os Montes": "T. Montes",
    "Douro": "Douro",
    "Bairrada": "Bairrada",
    "Dão": "Dão",
    "Dao": "Dão",
    "Beira Interior": "Beira Interior",
    "Távora-Varosa": "Távora-Varosa",
    "Tavora-Varosa": "Távora-Varosa",
    "Tejo": "Tejo",
    "Lisboa": "Lisboa",
    "Península de Setúbal": "P. Setúbal",
    "Peninsula de Setubal": "P. Setúbal",
    "Alentejo": "Alentejo",
    "Algarve": "Algarve",
    "Madeira": "Madeira",
    "Açores": "Açores",
    "Acores": "Açores",
}

REGIONS_TO_KEEP = [
    "Verdes",
    "T. Montes",
    "Douro",
    "Bairrada",
    "Dão",
    "Beira Interior",
    "Távora-Varosa",
    "Tejo",
    "Lisboa",
    "P. Setúbal",
    "Alentejo",
    "Algarve",
    "Madeira",
    "Açores",
]


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_for_matching(value):
    text = normalize_text(value)
    replacements = {
        "ã": "a",
        "á": "a",
        "à": "a",
        "â": "a",
        "ç": "c",
        "é": "e",
        "ê": "e",
        "í": "i",
        "ó": "o",
        "ô": "o",
        "õ": "o",
        "ú": "u",
        "Á": "A",
        "Ã": "A",
        "Ç": "C",
        "É": "E",
        "Í": "I",
        "Ó": "O",
        "Ú": "U",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


REGION_MATCH = {normalize_for_matching(raw): clean for raw, clean in REGION_MAP.items()}


def parse_year(value):
    if pd.isna(value):
        return None

    if isinstance(value, (pd.Timestamp, datetime)):
        return int(value.year)

    text = normalize_text(value)

    match = re.search(r"(19\d{2}|20\d{2})", text)
    if match:
        return int(match.group(1))

    return None


def clean_number(value):
    if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
        return float(value)

    text = normalize_text(value)
    if text == "" or text.lower() in {"nan", "none", "---"}:
        return np.nan

    text = re.sub(r"[^0-9,.-]", "", text)
    if text in {"", "-", ".", ","}:
        return np.nan

    has_comma = "," in text
    has_dot = "." in text

    if has_comma and has_dot:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif has_comma:
        text = text.replace(",", ".")
    elif has_dot and re.fullmatch(r"-?\d{1,3}(\.\d{3})+", text):
        text = text.replace(".", "")

    try:
        return float(text)
    except ValueError:
        return np.nan

## 5. Extract Vineyard Area Table

In [ ]:
header_row = None

for row_idx in range(len(raw_df)):
    row_values = [parse_year(value) for value in raw_df.iloc[row_idx].tolist()]
    year_count = sum(year is not None for year in row_values)
    first_cell = normalize_for_matching(raw_df.iat[row_idx, 0]).lower()

    if "regiao" in first_cell and year_count >= 5:
        header_row = row_idx
        break

if header_row is None:
    raise ValueError("Could not find the vineyard area header row.")

print("Header row:", header_row)

year_columns = {
    col_idx: parse_year(value)
    for col_idx, value in enumerate(raw_df.iloc[header_row].tolist())
    if parse_year(value) is not None
}

print("Years found:", sorted(set(year_columns.values())))

records = []

for row_idx in range(header_row + 1, len(raw_df)):
    raw_region = normalize_text(raw_df.iat[row_idx, 0])
    region_key = normalize_for_matching(raw_region)

    if region_key.lower() in {"total geral", "total - continente", "total - regioes autonomas"}:
        continue

    if region_key not in REGION_MATCH:
        continue

    region = REGION_MATCH[region_key]

    for col_idx, year_start in year_columns.items():
        vineyard_area_ha = clean_number(raw_df.iat[row_idx, col_idx])

        if pd.isna(vineyard_area_ha):
            continue

        records.append(
            {
                "region": region,
                "year_start": int(year_start),
                "vineyard_area_ha": vineyard_area_ha,
            }
        )

vineyard_area = pd.DataFrame(records)
vineyard_area = vineyard_area[vineyard_area["region"].isin(REGIONS_TO_KEEP)]
vineyard_area = vineyard_area.drop_duplicates(subset=["region", "year_start"], keep="first")
vineyard_area = vineyard_area.sort_values(["year_start", "region"]).reset_index(drop=True)
vineyard_area["vineyard_area_ha"] = vineyard_area["vineyard_area_ha"].round(2)

display(vineyard_area.head(20))
print("Clean dataset shape:", vineyard_area.shape)

## 6. Quality Checks

In [ ]:
print("Dataset shape:", vineyard_area.shape)

print("\nColumns:")
print(vineyard_area.columns.tolist())

print("\nYears:")
print(vineyard_area["year_start"].min(), "to", vineyard_area["year_start"].max())
print(sorted(vineyard_area["year_start"].unique()))

print("\nRegions:")
print(sorted(vineyard_area["region"].unique()))

print("\nDuplicated region-year rows:")
print(vineyard_area.duplicated(subset=["region", "year_start"]).sum())

print("\nMissing values:")
display(vineyard_area.isna().sum().to_frame("missing_count"))

print("\nSummary statistics for vineyard_area_ha:")
display(vineyard_area["vineyard_area_ha"].describe())

print("\nRows per region:")
display(vineyard_area.groupby("region")["year_start"].agg(["min", "max", "count"]).reset_index())

## 7. Save Processed CSV

In [ ]:
vineyard_area.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Saved processed dataset to:")
print(OUTPUT_FILE)

## 8. Optional Merge Check with Wine Production

This check is only to confirm that vineyard area can be used later as an explanatory variable for wine production models.

In [ ]:
PRODUCTION_FILE = ROOT_DIR / "data" / "processed" / "wine_production_by_region_clean.csv"

if PRODUCTION_FILE.exists():
    production = pd.read_csv(PRODUCTION_FILE)
    production["year_start"] = pd.to_numeric(production["year_start"], errors="coerce").astype("Int64")

    merged = production.merge(
        vineyard_area,
        on=["region", "year_start"],
        how="left",
    )

    print("Production shape:", production.shape)
    print("Merged shape:", merged.shape)
    print("Rows with missing vineyard_area_ha:", merged["vineyard_area_ha"].isna().sum())
    display(merged.head(20))
else:
    print("Production file not found yet. Run ivv_wine_production.ipynb first to test the merge.")

## 9. Download CSV

In [ ]:
from google.colab import files

files.download(str(OUTPUT_FILE))